In [96]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [97]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso, Ridge, ElasticNet
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, mean_absolute_error, root_mean_squared_error, mean_absolute_percentage_error, r2_score
import optuna

In [98]:
RANDOM_STATE = 42

In [99]:
data = pd.read_csv('winequality-white_finished.csv', sep=';')

In [100]:
X = data.drop(columns = ['quality'])
y = data['quality']

In [101]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.7)

In [102]:
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

Простая линейная регрессия + Optuna

In [ ]:

def objective_linear(trial):
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])

    linear_regression_model = LinearRegression(fit_intercept = fit_intercept)

    linear_regression_model.fit(X_train, y_train)

    LR_pred = linear_regression_model.predict(X_test)

    return mean_squared_error(y_test, LR_pred)


study = optuna.create_study(direction = 'minimize')
study.optimize(objective_linear, n_trials = 100)

best_model = LinearRegression(fit_intercept = study.best_params['fit_intercept'])
best_model.fit(X_train, y_train)

best_pred = best_model.predict(X_test)


[I 2026-03-30 12:43:28,270] A new study created in memory with name: no-name-921516c1-de69-40f3-9ea7-910ddfc4a8cd
[I 2026-03-30 12:43:28,275] Trial 0 finished with value: 0.5626395331856038 and parameters: {'fit_intercept': True}. Best is trial 0 with value: 0.5626395331856038.
[I 2026-03-30 12:43:28,278] Trial 1 finished with value: 35.1663108279593 and parameters: {'fit_intercept': False}. Best is trial 0 with value: 0.5626395331856038.
[I 2026-03-30 12:43:28,281] Trial 2 finished with value: 0.5626395331856038 and parameters: {'fit_intercept': True}. Best is trial 0 with value: 0.5626395331856038.
[I 2026-03-30 12:43:28,284] Trial 3 finished with value: 35.1663108279593 and parameters: {'fit_intercept': False}. Best is trial 0 with value: 0.5626395331856038.
[I 2026-03-30 12:43:28,286] Trial 4 finished with value: 35.1663108279593 and parameters: {'fit_intercept': False}. Best is trial 0 with value: 0.5626395331856038.
[I 2026-03-30 12:43:28,290] Trial 5 finished with value: 35.1663

In [104]:
print("Простая линейная регрессия + Optuna:")
print(f'MSE : {mean_squared_error(y_test, best_pred)}')
print(f'MAE : {mean_absolute_error(y_test, best_pred)}')
print(f'RMSE : {root_mean_squared_error(y_test, best_pred)}')
print(f'MAPE : {mean_absolute_percentage_error(y_test, best_pred)}')
print(f'R2 : {r2_score(y_test, best_pred)}')

Простая линейная регрессия + Optuna:
MSE : 0.5626395331856038
MAE : 0.5853674740820807
RMSE : 0.7500930163557076
MAPE : 0.10236539786646687
R2 : 0.2832946499873149


Линейная регрессия + L1 + Optuna

In [ ]:

def obj_lin_lasso(trial):
    fit_intercept = trial.suggest_categorical('fit_intercept', [True, False])
    alpha = trial.suggest_float('alpha', 1e-5, 1e2, log = True)
    tol = trial.suggest_float('tol', 1e-8, 1e-3, log = True)

    linear_regression_lasso_model = Lasso(fit_intercept = fit_intercept, alpha = alpha, tol = tol, random_state = RANDOM_STATE)

    linear_regression_lasso_model.fit(X_train, y_train)
    Lasso_LR_pred = linear_regression_lasso_model.predict(X_test)

    return mean_squared_error(y_test, Lasso_LR_pred)

study = optuna.create_study(direction='minimize')
study.optimize(obj_lin_lasso, n_trials=100)

best_model_lasso = Lasso(fit_intercept=study.best_params['fit_intercept'], alpha = study.best_params['alpha'], tol = study.best_params['tol'])
best_model_lasso.fit(X_train, y_train)

Lasso_LR_pred = best_model_lasso.predict(X_test)




[I 2026-03-30 12:43:28,699] A new study created in memory with name: no-name-5672e7f4-137a-4cf7-8575-59b31b90faa6
[I 2026-03-30 12:43:28,701] Trial 0 finished with value: 0.582649901008684 and parameters: {'fit_intercept': True, 'alpha': 0.034893673569445754, 'tol': 1.1176917846578099e-08}. Best is trial 0 with value: 0.582649901008684.
[I 2026-03-30 12:43:28,704] Trial 1 finished with value: 35.35013979596011 and parameters: {'fit_intercept': False, 'alpha': 0.2251935167718735, 'tol': 0.0005294153862925137}. Best is trial 0 with value: 0.582649901008684.
[I 2026-03-30 12:43:28,709] Trial 2 finished with value: 0.57045955110742 and parameters: {'fit_intercept': True, 'alpha': 0.010196026927045992, 'tol': 1.7647835844340924e-05}. Best is trial 2 with value: 0.57045955110742.
[I 2026-03-30 12:43:28,715] Trial 3 finished with value: 35.16630897874001 and parameters: {'fit_intercept': False, 'alpha': 2.333093080831789e-05, 'tol': 0.0008312953823284915}. Best is trial 2 with value: 0.570459

In [106]:
print("Линейная регрессия + L1 регуляризация:")
print(f'MSE : {mean_squared_error(y_test, Lasso_LR_pred)}')
print(f'MAE : {mean_absolute_error(y_test, Lasso_LR_pred)}')
print(f'RMSE : {root_mean_squared_error(y_test, Lasso_LR_pred)}')
print(f'MAPE : {mean_absolute_percentage_error(y_test, Lasso_LR_pred)}')
print(f'R2 : {r2_score(y_test, Lasso_LR_pred)}')

Линейная регрессия + L1 регуляризация:
MSE : 0.5622370695419625
MAE : 0.5853725728266904
RMSE : 0.7498246925395046
MAPE : 0.10237392261406443
R2 : 0.28380731898686085


Линейная регрессия + L2 + GSCV

In [107]:

linear_regression_ridge_model = Ridge()

linear_regression_ridge_model.fit(X_train, y_train)
Ridge_LR_pred = linear_regression_ridge_model.predict(X_test)


print("Линейная регрессия + L2 регуляризация:")
print(f'MSE : {mean_squared_error(y_test, Ridge_LR_pred)}')
print(f'MAE : {mean_absolute_error(y_test, Ridge_LR_pred)}')
print(f'RMSE : {root_mean_squared_error(y_test, Ridge_LR_pred)}')
print(f'MAPE : {mean_absolute_percentage_error(y_test, Ridge_LR_pred)}')
print(f'R2 : {r2_score(y_test, Ridge_LR_pred)}')


Линейная регрессия + L2 регуляризация:
MSE : 0.5624692946636193
MAE : 0.5853333037414
RMSE : 0.7499795294963851
MAPE : 0.10236137477670752
R2 : 0.28351150438926864


Линейная регрессия + ElasticNet + RSCV

In [108]:

linear_regression_elasticNet_model = ElasticNet()
linear_regression_elasticNet_model.fit(X_train, y_train)
ElasticNet_LR_pred = linear_regression_elasticNet_model.predict(X_test)

print("Линейная регрессия + ElasticNet регуляризация:")
print(f'MSE : {mean_squared_error(y_test, ElasticNet_LR_pred)}')
print(f'MAE : {mean_absolute_error(y_test, ElasticNet_LR_pred)}')
print(f'RMSE : {root_mean_squared_error(y_test, ElasticNet_LR_pred)}')
print(f'MAPE : {mean_absolute_percentage_error(y_test, ElasticNet_LR_pred)}')
print(f'R2 : {r2_score(y_test, ElasticNet_LR_pred)}')


Линейная регрессия + ElasticNet регуляризация:
MSE : 0.7864483605767993
MAE : 0.676712722005733
RMSE : 0.8868192378251609
MAPE : 0.11949897042689649
R2 : -0.0017990459055776675


Линейная регрессия + PolynomialFeatures + Optuna

In [109]:


Input =  [('standardscaler', StandardScaler()), ('polynomial', PolynomialFeatures(degree=2, include_bias=False)), ('model', LinearRegression())]

pipe = Pipeline(Input)

pipe.fit(X_train, y_train)

polynomial_LR_pred = pipe.predict(X_test)

print("Линейная регрессия + полиномиальные признаки:")
print(f'MSE : {mean_squared_error(y_test, polynomial_LR_pred)}')
print(f'MAE : {mean_absolute_error(y_test, polynomial_LR_pred)}')
print(f'RMSE : {root_mean_squared_error(y_test, polynomial_LR_pred)}')
print(f'MAPE : {mean_absolute_percentage_error(y_test, polynomial_LR_pred)}')
print(f'R2 : {r2_score(y_test, polynomial_LR_pred)}')

Линейная регрессия + полиномиальные признаки:
MSE : 0.5548844432771355
MAE : 0.5781486311487029
RMSE : 0.7449056606558548
MAPE : 0.10121514768412992
R2 : 0.2931732918164077


Метрики

In [110]:

def self_calc_metrics(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    mse = np.mean((y_true - y_pred)**0.5)
    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(mse)
    mape = np.mean(np.abs((y_true - y_pred)/y_true)) * 100
    r2 = 1 - ((np.sum((y_true - y_pred)**2))/ np.sum((y_true - np.mean(y_true))**2))

    return mse, mae, rmse, mape, r2